# 1. Цикл for

- NumPy дает возможность применять операции целиком к массиву
- Несмотря на это, обработка больших данных зачастую возможна только порциями ("батчами")
- Циклы - база для потоковой обработки данных, батчевой загрузки данных в модель и для обработки больших csv в Pandas

По объекту list можно совершать итерации

In [1]:
customers = ["MARY SMITH", "PATRICIA JOHNSON", "LINDA WILLIAMS"]

for customer in customers:
    print(customer)

MARY SMITH
PATRICIA JOHNSON
LINDA WILLIAMS


А по объекту int итерации совершать нельзя

In [2]:
last_rental_id = 16049

for x in last_rental_id:
    print(x)

TypeError: 'int' object is not iterable

Почему? Что находится внутри объектов, которые могут использоваться для цикла for?

Как Python понимает, что список можно перебирать в цикле, а число нельзя?

# 2. Итерируемый объект (Iterable) и Итератор (Iterator)

**Iterable** - объект, из которого можно получить итератор
- Имеет метод \_\_iter\_\_()
- Примеры: list, string, dict

**Iterator** - объект, который отдает элементы по одному
- Имеет методы \_\_iter\_\_() и \_\_next\_\_()
- Итератор создается из итерируемого объекта функцией iter()

Протокол итератора: \_\_iter\_\_ и \_\_next\_\_:

- iter(объект) => вызывает объект.\_\_iter\_\_() => возвращает итератор

- next(итератор) => вызывает итератор.\_\_next\_\_() => возвращает следующий элемент

- Когда элементы закончились, \_\_next\_\_() возбуждает StopIteration

- Цикл for перехватывает это исключение автоматически

In [3]:
categories = ["Action", "Comedy", "Drama", "Documentary"] # Это Iterable
it = iter(categories)                                     # Это Iterator

In [4]:
type(categories)

list

In [5]:
type(it)

list_iterator

In [6]:
print(next(it)) # Вызываем следующий элемент итератора

Action


In [7]:
print(next(it)) # Вызываем следующий элемент итератора

Comedy


In [8]:
print(next(it)) # Вызываем следующий элемент итератора

Drama


In [9]:
print(next(it)) # Вызываем следующий элемент итератора

Documentary


In [11]:
print(next(it)) # Вызываем следующий элемент итератора и получаем окончание итераций

StopIteration: 

# 3. Анатомия цикла for

In [12]:
categories = ["Action", "Comedy", "Drama", "Documentary"]
it = iter(categories)
while True:
    try:
        item = next(it)
    except StopIteration:
        break
    print(item)

Action
Comedy
Drama
Documentary


# 4. Примеры итераторов из разных объектов

## 4.1. Итерация по списку

In [13]:
# Результат SQL-запроса SELECT customer_id, amount FROM payment LIMIT 5 в виде кортежей
payments = [(1, 2.99), (1, 0.99), (2, 5.99), (3, 4.99), (2, 1.99)]

In [14]:
it = iter(payments)

In [15]:
print(next(it))

(1, 2.99)


In [16]:
print(next(it))

(1, 0.99)


## 4.2. Итерация по строке

In [17]:
# Строка — название фильма
title = "ACADEMY DINOSAUR"

In [18]:
it_title = iter(title)

In [19]:
print(next(it_title))

A


In [20]:
print(next(it_title))

C


In [21]:
print(next(it_title))

A


## 4.3. Итерация по словарю

In [22]:
# Словарь — строка из таблицы customer
customer = {"customer_id": 1, "first_name": "MARY", "last_name": "SMITH"}

In [23]:
it_customer = iter(customer)

In [24]:
print(next(it_customer))

customer_id


In [25]:
print(next(it_customer))

first_name


# 5. Создание собственных итераторов

## 5.1. Обратный отсчет

In [26]:
class DueDateCountdown:
    """Считает дни, оставшиеся до истечения срока аренды (rental_duration)"""
    def __init__(self, rental_duration):
        self.days_left = rental_duration

    def __iter__(self):
        return self

    def __next__(self):
        if self.days_left <= 0:
            raise StopIteration
        current = self.days_left
        self.days_left -= 1
        return current

In [27]:
# У фильма ACADEMY DINOSAUR rental_duration = 6 дней
for day in DueDateCountdown(6):
    print(f"Осталось дней до возврата: {day}")

Осталось дней до возврата: 6
Осталось дней до возврата: 5
Осталось дней до возврата: 4
Осталось дней до возврата: 3
Осталось дней до возврата: 2
Осталось дней до возврата: 1


In [28]:
it_due_date = DueDateCountdown(5)
print(next(it_due_date))
print(next(it_due_date))
print(next(it_due_date))

5
4
3


## 5.2. Список актеров в обратном порядке

In [30]:
class ReverseActorIterator:
    """Перебирает список актеров фильма от последнего к первому."""
    def __init__(self, actors):
        self.actors = actors
        self.index = len(actors)

    def __iter__(self):
        return self

    def __next__(self):
        if self.index == 0:
            raise StopIteration
        self.index -= 1
        return self.actors[self.index]

In [31]:
# Актеры фильма ACADEMY DINOSAUR
actors = ["PENELOPE GUINESS", "CHRISTIAN GABLE", "LUCILLE TRACY"]
for actor in ReverseActorIterator(actors):
    print(actor)

LUCILLE TRACY
CHRISTIAN GABLE
PENELOPE GUINESS


## 5.3. Перебор аренд и выбор тех, которые не имеют даты возврата

In [33]:
class OverdueRentalIterator:
    """Отдает только аренды без даты возврата (return_date is None) —
    аналог WHERE return_date IS NULL в SQL."""
    def __init__(self, rentals):
        self.rentals = rentals
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.index < len(self.rentals):
            rental = self.rentals[self.index]
            self.index += 1
            if rental["return_date"] is None:
                return rental
        raise StopIteration

In [34]:
rentals = [
    {"rental_id": 1, "customer_id": 130, "return_date": "2024-01-05"},
    {"rental_id": 2, "customer_id": 45,  "return_date": None},
    {"rental_id": 3, "customer_id": 87,  "return_date": None},
    {"rental_id": 4, "customer_id": 12,  "return_date": "2024-01-10"},
]

In [35]:
for r in OverdueRentalIterator(rentals):
    print(r["rental_id"], "customer_id:", r["customer_id"])

2 customer_id: 45
3 customer_id: 87


Проблемы ручных итераторов: 
- много кода на простую задачу
- классы-итераторы требуют: \_\_init\_\_, ручное хранение состояния, \_\_iter\_\_, \_\_next\_\_, ручной raise StopIteration
- для простой задачи писать целый класс избыточно

Более простой спокоб - это генераторы

# 6. Генераторы

**Генератор** - функция, содержащая yield вместо (или вместе с) return

- Вызов такой функции не выполняет код сразу
- Возвращается объект-генератор, уже автоматически реализующий \_\_iter\_\_ и \_\_next\_\_
- Код выполняется по частям, при каждом вызове next() до следующего yield
- Функция «замораживается» после yield и продолжает с этого места при следующем вызове

In [36]:
def due_date_countdown(rental_duration):
    print("Отсчет запущен")
    days_left = rental_duration
    while days_left > 0:
        yield days_left
        days_left -= 1

In [37]:
gen = due_date_countdown(6)

In [38]:
print(gen)

<generator object due_date_countdown at 0x0000019541532810>


In [39]:
print(type(gen))

<class 'generator'>


In [40]:
print(next(gen))

Отсчет запущен
6


In [41]:
print(next(gen))

5


In [42]:
print(next(gen))

4


In [46]:
print(next(gen))

StopIteration: 

# 7. Создание собственных генераторов

## 7.1. Перебор аренд и выбор тех, которые не имеют даты возврата

In [47]:
def overdue_rentals(rentals):
    for rental in rentals:
        if rental["return_date"] is None:
            yield rental

In [49]:
rentals = [
    {"rental_id": 1, "customer_id": 130, "return_date": "2024-01-05"},
    {"rental_id": 2, "customer_id": 45,  "return_date": None},
    {"rental_id": 3, "customer_id": 87,  "return_date": None},
    {"rental_id": 4, "customer_id": 12,  "return_date": "2024-01-10"},
]

In [51]:
for r in overdue_rentals(rentals):
    print(r["rental_id"], "customer_id:", r["customer_id"])

2 customer_id: 45
3 customer_id: 87


## 7.2. Перебор фильмов по одному

In [52]:
def fetch_films_page_by_page(films, page_size=2):
    """Имитирует запрос: SELECT * FROM film LIMIT page_size OFFSET n."""
    offset = 0
    while offset < len(films):
        print(f"-- запрос к БД: LIMIT {page_size} OFFSET {offset}")
        page = films[offset:offset + page_size]
        yield page
        offset += page_size

In [53]:
films = ["ACADEMY DINOSAUR", "ACE GOLDFINGER", "ADAPTATION HOLES", "AFFAIR PREJUDICE"]
for page in fetch_films_page_by_page(films, page_size=2):
    print("Обрабатываем страницу:", page)

-- запрос к БД: LIMIT 2 OFFSET 0
Обрабатываем страницу: ['ACADEMY DINOSAUR', 'ACE GOLDFINGER']
-- запрос к БД: LIMIT 2 OFFSET 2
Обрабатываем страницу: ['ADAPTATION HOLES', 'AFFAIR PREJUDICE']


# 8. Генераторные выражения

- Если нужно передать последовательность в функцию sum(), any(), all(), max(), min(), join()
- Эта функция пройдет по всем элементам один раз
- Создавать для нее промежуточный список - потеря памяти
- Список полностью строится в памяти, а затем сразу выбрасывается сборщиком мусора

Решение: генераторные выражения

Генераторное выражение нельзя использовать, если:
- нужно пройти по данным больше одного раза (генератор исчерпывается после первого прохода)
- нужен доступ по индексу (data[3]) или len(data)
- нужно передать данные туда, где ожидается именно список/коллекция

## 8.1. Создание генераторного выражения вместо списка

In [54]:
# List comprehension — создает список сразу
titles_list = [t.upper() for t in ["matrix", "inception", "avatar"]]

In [56]:
titles_list = []
for t in ["matrix", "inception", "avatar"]:
    titles_list.append(t.upper())

In [57]:
titles_list

['MATRIX', 'INCEPTION', 'AVATAR']

In [58]:
# Generator expression — создает генератор (ленивый)
titles_gen = (t.upper() for t in ["matrix", "inception", "avatar"])

In [59]:
titles_gen

<generator object <genexpr> at 0x00000195415332A0>

In [60]:
for i in titles_gen:
    print(i)

MATRIX
INCEPTION
AVATAR


## 8.2. Генераторные выражения со встроенными функциями

In [61]:
payments = [
    {"customer_id": 1, "amount": 4.99, "store_id": 1},
    {"customer_id": 2, "amount": 0.99, "store_id": 2},
    {"customer_id": 1, "amount": 5.99, "store_id": 1},
    {"customer_id": 3, "amount": 2.99, "store_id": 2},
]

In [62]:
# sum() — общая выручка магазина 1
store1_total = sum(p["amount"] for p in payments if p["store_id"] == 1)
print(store1_total)

10.98


In [64]:
type(store1_total)

float

In [65]:
# any() — есть ли платежи выше 5 долларов
has_big_payment = any(p["amount"] > 5 for p in payments)
print(has_big_payment)

True


In [66]:
# all() — все ли платежи из магазина 1
all_store1 = all(p["store_id"] == 1 for p in payments)
print(all_store1)

False


In [67]:
# max() с генераторным выражением и ключом
biggest_payment = max(payments, key=lambda p: p["amount"])
print(biggest_payment["customer_id"])

1


# 9. yield from

**yield from <итерируемый_объект>** - это способ делегировать генерацию значений другому итератору/генератору, не переписывая цикл for ... yield вручную

_yield from other_gen_ является сокращенной записью для:

_for item in other_gen:_

_yield item_

In [68]:
def rentals_store_1():
    yield {"rental_id": 1, "store_id": 1}
    yield {"rental_id": 2, "store_id": 1}

def rentals_store_2():
    yield {"rental_id": 3, "store_id": 2}
    yield {"rental_id": 4, "store_id": 2}

def all_rentals():
    yield from rentals_store_1()
    yield from rentals_store_2()

In [75]:
for r in all_rentals():
    print(r)

{'rental_id': 1, 'store_id': 1}
{'rental_id': 2, 'store_id': 1}
{'rental_id': 3, 'store_id': 2}
{'rental_id': 4, 'store_id': 2}


In [83]:
def flatten(nested):
    """Разворачивает произвольно вложенные списки в плоский поток"""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

In [84]:
actors_by_film = [
    ["PENELOPE GUINESS", "CHRISTIAN GABLE"],
    [["BOB FAWCETT"], "CUBA OLIVIER"],
    "NICK WAHLBERG",
]

In [86]:
print(list(flatten(actors_by_film)))

['PENELOPE GUINESS', 'CHRISTIAN GABLE', 'BOB FAWCETT', 'CUBA OLIVIER', 'NICK WAHLBERG']


# 10. itertools

- **itertools** - модуль стандартной библиотеки Python, содержащий готовые строительные блоки для работы с итераторами

- Все функции модуля возвращают итераторы, а не списки, то есть работают лениво, не потребляя память на весь результат сразу

- Вместо того чтобы каждый раз писать свой генератор для типовой задачи (бесконечный счетчик, объединение потоков, группировка, комбинаторика), взять уже написанную, оптимизированную и протестированную функцию

In [54]:
import itertools
print(dir(itertools))

['__doc__', '__loader__', '__name__', '__package__', '__spec__', '_grouper', '_tee', '_tee_dataobject', 'accumulate', 'batched', 'chain', 'combinations', 'combinations_with_replacement', 'compress', 'count', 'cycle', 'dropwhile', 'filterfalse', 'groupby', 'islice', 'pairwise', 'permutations', 'product', 'repeat', 'starmap', 'takewhile', 'tee', 'zip_longest']


**Три категории функций itertools**

1. Бесконечные итераторы (infinite iterators)
   
Генерируют значения бесконечно, пока их не остановят вручную (islice, break, условие).

- count() — бесконечный счетчик
- cycle() — бесконечное повторение последовательности
- repeat() — бесконечное повторение одного значения

2. Итераторы, завершающиеся по кратчайшему входу (terminating iterators)

Работают с конечными входными данными и останавливаются, когда данные заканчиваются.

- chain(), chain.from_iterable() — объединение потоков
- islice() — «срез» итератора
- groupby() — группировка соседних элементов
- accumulate() — накопительная сумма/операция
- compress(), filterfalse(), dropwhile(), takewhile() — фильтрация
- tee() — размножение итератора на несколько копий
- zip_longest() — параллельный обход с выравниванием по самому длинному

3. Комбинаторные итераторы (combinatoric iterators)
   
Генерируют комбинации, перестановки, произведения элементов.

- product() — декартово произведение
- permutations() — перестановки
- combinations() — сочетания
- combinations_with_replacement() — сочетания с повторениями

## 10.1. Бесконечные итераторы

### 10.1.1. Count

In [87]:
from itertools import count, islice

In [108]:
ids = count(start=1000, step=1)

In [109]:
type(ids)

itertools.count

In [110]:
print(ids)

count(1000)


In [111]:
print(list(islice(ids, 5)))

[1000, 1001, 1002, 1003, 1004]


In [112]:
print(list(islice(ids, 3)))

[1005, 1006, 1007]


In [113]:
# Присвоение новых rental_id при пакетной вставке
last_rental_id = 16049
new_rental_ids = count(start=last_rental_id + 1, step = 1)

new_rentals = [("BLADE RUNNER", 130), ("MATRIX RELOADED", 45), ("INCEPTION CLONE", 87)]
for rental_id, (title, customer_id) in zip(new_rental_ids, new_rentals):
    print(f"rental_id={rental_id}, film={title}, customer_id={customer_id}")

rental_id=16050, film=BLADE RUNNER, customer_id=130
rental_id=16051, film=MATRIX RELOADED, customer_id=45
rental_id=16052, film=INCEPTION CLONE, customer_id=87


### 10.1.2. Cycle

In [114]:
from itertools import cycle, islice

In [115]:
pattern = cycle(["G", "PG", "R"])
print(list(islice(pattern, 7)))

['G', 'PG', 'R', 'G', 'PG', 'R', 'G']


In [116]:
# В dvdrental всего два магазина: store_id 1 и 2
stores = cycle([1, 2])
new_films = ["ACADEMY DINOSAUR", "ACE GOLDFINGER", "ADAPTATION HOLES", "AFFAIR PREJUDICE", "AFRICAN EGG"]

for film, store_id in zip(new_films, stores):
    print(f"{film} -> store_id={store_id}")

ACADEMY DINOSAUR -> store_id=1
ACE GOLDFINGER -> store_id=2
ADAPTATION HOLES -> store_id=1
AFFAIR PREJUDICE -> store_id=2
AFRICAN EGG -> store_id=1


## 10.2. Терминирующие итераторы

### 10.2.1. Chain

In [118]:
from itertools import chain

In [120]:
action_films = ["AIRPORT POLLOCK", "ALAMO VIDEOTAPE", "AMISTAD MIDSUMMER"]
comedy_films = ["ACE GOLDFINGER", "AFFAIR PREJUDICE"]
drama_films = ["ALASKA PHANTOM"]

all_films = chain(action_films, comedy_films, drama_films)

In [121]:
all_films

In [122]:
for film in all_films:
    print(film)

AIRPORT POLLOCK
ALAMO VIDEOTAPE
AMISTAD MIDSUMMER
ACE GOLDFINGER
AFFAIR PREJUDICE
ALASKA PHANTOM


In [123]:
# chain для объединения генераторов платежей по двум магазинам
def payments_store_1():
    yield ("P-1", 1, 4.99)
    yield ("P-2", 1, 2.99)

def payments_store_2():
    yield ("P-3", 2, 5.99)
    yield ("P-4", 2, 1.99)

total = sum(amount for _, _, amount in chain(payments_store_1(), payments_store_2()))
print(f"Общая сумма платежей по двум магазинам: {total}")

Общая сумма платежей по двум магазинам: 15.96


### 10.2.2. Groupby

In [124]:
from itertools import groupby

In [125]:
films = [
    {"title": "ACADEMY DINOSAUR", "category": "Documentary"},
    {"title": "ACE GOLDFINGER", "category": "Horror"},
    {"title": "ADAPTATION HOLES", "category": "Documentary"},
    {"title": "AFFAIR PREJUDICE", "category": "Horror"},
    {"title": "AFRICAN EGG", "category": "Family"},
]

In [126]:
# groupby требует, чтобы данные были предварительно отсортированы по ключу группировки
films_sorted = sorted(films, key=lambda f: f["category"])
print(films_sorted)

[{'title': 'ACADEMY DINOSAUR', 'category': 'Documentary'}, {'title': 'ADAPTATION HOLES', 'category': 'Documentary'}, {'title': 'AFRICAN EGG', 'category': 'Family'}, {'title': 'ACE GOLDFINGER', 'category': 'Horror'}, {'title': 'AFFAIR PREJUDICE', 'category': 'Horror'}]


In [127]:
for category, group in groupby(films_sorted, key=lambda f: f["category"]):
    titles = [f["title"] for f in group]
    print(category, "->", titles)

Documentary -> ['ACADEMY DINOSAUR', 'ADAPTATION HOLES']
Family -> ['AFRICAN EGG']
Horror -> ['ACE GOLDFINGER', 'AFFAIR PREJUDICE']


### 10.2.3. Accumulate

In [128]:
from itertools import accumulate

In [129]:
# accumulate — накопительная выручка по дням (для графика динамики платежей)
daily_revenue = [45.5, 32.0, 61.2, 28.9, 55.0]
running_total = list(accumulate(daily_revenue))
print(running_total)

[45.5, 77.5, 138.7, 167.6, 222.6]


## 10.3. Комбинаторные итераторы

### 10.3.1. Product

In [130]:
from itertools import product

In [131]:
ratings = ["G", "PG", "R"]
stores = [1, 2]

# Все возможные комбинации рейтинг+магазин — например, для генерации тестовых сценариев
combos = list(product(ratings, stores))
print(combos)

[('G', 1), ('G', 2), ('PG', 1), ('PG', 2), ('R', 1), ('R', 2)]


In [132]:
# Эквивалент через вложенные циклы (то, что product заменяет):
combos_manual = [(r, s) for r in ratings for s in stores]
combos_manual

[('G', 1), ('G', 2), ('PG', 1), ('PG', 2), ('R', 1), ('R', 2)]

### 10.3.2. Permutations

Все возможные упорядоченные перестановки

In [133]:
from itertools import permutations

In [135]:
actors = ["PENELOPE GUINESS", "NICK WAHLBERG", "ED CHASE"]
print(list(permutations(actors)))

[('PENELOPE GUINESS', 'NICK WAHLBERG', 'ED CHASE'), ('PENELOPE GUINESS', 'ED CHASE', 'NICK WAHLBERG'), ('NICK WAHLBERG', 'PENELOPE GUINESS', 'ED CHASE'), ('NICK WAHLBERG', 'ED CHASE', 'PENELOPE GUINESS'), ('ED CHASE', 'PENELOPE GUINESS', 'NICK WAHLBERG'), ('ED CHASE', 'NICK WAHLBERG', 'PENELOPE GUINESS')]


### 10.3.3. Combinations

Все возможные сочетания заданной длины, без учета порядка

In [136]:
from itertools import combinations

In [137]:
actors = ["PENELOPE GUINESS", "NICK WAHLBERG", "ED CHASE"]
print(list(combinations(actors, 2)))

[('PENELOPE GUINESS', 'NICK WAHLBERG'), ('PENELOPE GUINESS', 'ED CHASE'), ('NICK WAHLBERG', 'ED CHASE')]
